# REM Sleep Behavior Disorder - Model Training Pipeline

This notebook orchestrates the complete training pipeline for REM classification models.

In [ ]:
# Import required libraries
import sys
import os
import numpy as np
import pandas as pd
from typing import Dict, Tuple, Optional
import json
import warnings

warnings.filterwarnings('ignore')

# Add parent directory to path for imports
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('.')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print("Imports completed successfully")

In [ ]:
# Import from project modules
from dataset.preprocess import prepare_dataset
from models.xgb_model import XGBModel
from models.rf_model import RFModel
from models.fuser import REMFuser
from models.model_output import ModelOutput, ModalityResult
from models.train import REMTrainingPipeline

print("Module imports completed successfully")

## Step 1: Initialize Training Pipeline

In [ ]:
# Initialize pipeline
data_dir = '../dataset'
output_dir = './output'

pipeline = REMTrainingPipeline(data_dir=data_dir, output_dir=output_dir)
print(f"Pipeline initialized")
print(f"  Data directory: {data_dir}")
print(f"  Output directory: {output_dir}")

## Step 2: Load and Prepare Data

In [ ]:
# Load and prepare data
features, labels = pipeline.load_and_prepare_data()
print(f"\nData preparation completed")
print(f"Features shape: {features.shape}")
print(f"Labels distribution:\n{labels.value_counts()}")

## Step 3: Select REM Features

In [ ]:
# Select REM features and create test split
selected_features = pipeline.select_rem_features()
print(f"\nREM feature selection completed")
print(f"Training set: {pipeline.features.shape}")
print(f"Test set: {pipeline.test_features.shape}")

## Step 4: Train XGBoost Model

In [ ]:
# Train XGBoost model
xgb_output = pipeline.train_xgb_model()
print(f"\nXGBoost model training completed")
print(f"Accuracy: {xgb_output.accuracy:.4f}")
print(f"F1 Score: {xgb_output.f1_score:.4f}")

## Step 5: Train Random Forest Model

In [ ]:
# Train Random Forest model
rf_output = pipeline.train_rf_model()
print(f"\nRandom Forest model training completed")
print(f"Accuracy: {rf_output.accuracy:.4f}")
print(f"F1 Score: {rf_output.f1_score:.4f}")

## Step 6: Fuse Models and Generate Final Ensemble

In [ ]:
# Fuse models and generate final results
fusion_method = 'voting'  # Options: 'voting', 'weighted_voting', 'averaging', 'weighted_averaging'
modality_result = pipeline.fuse_and_evaluate(fusion_method=fusion_method)
print(f"\nModel fusion completed with {fusion_method} method")
print(f"Ensemble Accuracy: {modality_result.ensemble_accuracy:.4f}")
print(f"Ensemble F1 Score: {modality_result.ensemble_f1:.4f}")

## Step 7: Generate Training Report

In [ ]:
# Generate comprehensive training report
report = pipeline.generate_report()
print(f"\nTraining report generated")
print(f"Report saved to: {os.path.join(output_dir, 'training_report.txt')}")
print("\n" + "="*60)
print(report)
print("="*60)

## Step 8: Summary

In [ ]:
# Display final summary
print("\n" + "="*60)
print("TRAINING PIPELINE COMPLETED SUCCESSFULLY")
print("="*60)
print(f"\nOutput files created:")
for file in os.listdir(output_dir):
    file_path = os.path.join(output_dir, file)
    if os.path.isfile(file_path):
        size = os.path.getsize(file_path) / 1024
        print(f"  ✓ {file} ({size:.2f} KB)")
    else:
        print(f"  ✓ {file}/ (directory)")

print(f"\nKey Artifacts:")
print(f"  - Ensemble Model: {os.path.join(output_dir, 'rem_ensemble.pkl')}")
print(f"  - Results JSON: {os.path.join(output_dir, 'modality_result.json')}")
print(f"  - Report: {os.path.join(output_dir, 'training_report.txt')}")